In [ ]:
import sys
sys.path.insert(0,"../")
from data import get_data, get_site_ids, aggregate_by_interval

In [ ]:
uid = "WQS0038"
wdf = get_data(site_uid=uid).water

# resample taking the max value every day
# add datetime back as a column
# add new integer columns corresponding to months and days
# add another column corresponding to years
daily = wdf.resample("1D").agg("max").reset_index()
daily = daily.assign(dayofyear=daily["datetime"].dt.dayofyear)
daily = daily.assign(year=daily["datetime"].dt.year)

# groupby unique dayofyear (1-366) values and take mean
# restore the the dayofyear as a column
climate_avg = daily.groupby("dayofyear")["nitrate_con"].agg("mean").reset_index()
print(climate_avg.columns)
print(climate_avg.head())
print(climate_avg.shape)

In [ ]:
from data.rain import make_rain
print(make_rain.dist_to_sensor("WQS0039", 31))
print(make_rain.dist_to_sensor("WQS0039", 44))

In [ ]:
uid = "WQS0102"
wdf = get_data(site_uid=uid)
print(wdf.grid)
print(wdf.surplus.columns)

In [ ]:
import plotly.graph_objects as go
import plotly.colors as pc


def plot_water(fig):
    years = sorted(daily["year"].unique())
    n = len(years)
    palette = pc.sample_colorscale("Viridis", [i / (n - 1) for i in range(n)])
    color_map = dict(zip(years, palette))

    for yr, df in daily.groupby("year"):
        fig.add_trace(
            go.Scatter(
                x=df["dayofyear"],
                y=df["nitrate_con"],
                line=dict(color="#0A5B12", width=1.5),
                opacity=0.5,
                name=str(yr)
            )
        )
    fig.add_trace(
        go.Scatter(
            x=climate_avg["dayofyear"], 
            y=climate_avg["nitrate_con"],
            line=dict(color="#6909AA", width=2),
            name="mean",
            mode="lines"
            )
        )
    fig.update_layout(
            yaxis={"title": "Nitrate (mg/L)"},
            xaxis={"title": None},
            legend={"orientation": "h", "y": -0.15},
            margin={"t": 20, "b": 40, "l": 50, "r": 50},
        )
    return fig

rdf = get_data(uid).rain

rdf = rdf.groupby("date").agg("sum").reset_index()[["date", "precip_in_1d"]]
rdf = rdf.assign(dayofyear=rdf["date"].dt.dayofyear, year=rdf["date"].dt.year)
rain_mean = rdf.groupby("dayofyear").agg("mean").reset_index()[["dayofyear", "precip_in_1d"]]

def plot_rain(fig):
    for yr, df in rdf.groupby("year"):
        fig.add_trace(
            go.Scatter(
                x=df.dayofyear,
                y=df.precip_in_1d,
                name=f"Daily Rain {yr}",
                line=dict(color="#3993DC",width=1.5),
                opacity=0.5,
                yaxis="y2",
            )
        )

    fig.add_trace(
        go.Scatter(
            x=rain_mean.dayofyear,
            y=rain_mean.precip_in_1d,
            name="Precip mean",
            yaxis="y2",
            line=dict(color="#B10029",width=2),
            opacity=1
        )
    )
    fig.update_layout(
        yaxis2=dict(
            title="Precipitation (in)",
            overlaying="y",
            side="right",
            showgrid=False
        )
    )
    return fig
    
fig = go.Figure()

# comment out these lines to control what is graphed
fig = plot_water(fig)
fig = plot_rain(fig)
fig.show()

Apparently corn is planted in April - early May. We get our first spike day ~75 or ~80, I wonder why, maybe you fertilize in March? What's up with nitrogen spiking in March?